# Testing Agents with Microsoft Agent Framework
This notebook creates and runs agents using the **Microsoft Agent Framework** (`agent-framework`) SDK,
which provides a simple `Agent` + `FoundryChatClient` pattern with `agent.run()` for inference.

Best way to set it up is with [uv](https://docs.astral.sh/uv/)

1. in `agents_v2_maf` directory run `uv sync` (if python is missing, install it using `uv python install`)
2. in VSCode press [Ctrl+Shift+P] and select `Python: Select Interpreter`, choose the one from `agents_v2_maf` directory: `./agents_v2_maf/.venv/bin/python3.xx`

## Step 1 - Setup Environment and Initialize Clients
Create the `FoundryChatClient` using the Microsoft Agent Framework SDK (`agent-framework-foundry`)

In [ ]:
import atexit
import asyncio
import os
from azure.identity.aio import DefaultAzureCredential, AzureDeveloperCliCredential
from azure.ai.projects.aio import AIProjectClient
from agent_framework import Agent
from agent_framework_foundry import FoundryChatClient
from dotenv import load_dotenv

# Load environment variables from the .env file
load_dotenv(override=True)

print("🚀 Initializing Microsoft Agent Framework Testing...")
print("=" * 60)

project_endpoint = os.environ.get("AZURE_AI_FOUNDRY_CONNECTION_STRING")
deployment_name = os.environ.get("AZURE_OPENAI_CHAT_DEPLOYMENT_NAME")
tenant_id = os.environ.get("AZURE_TENANT_ID", None)

print("\n📋 Configuration:")
print(f"   📍 Endpoint: {project_endpoint[:50]}..." if project_endpoint else "   ⚠️ Endpoint: Not set")
print(f"   🤖 Deployment: {deployment_name}" if deployment_name else "   ⚠️ Deployment: Not set")

if not project_endpoint:
    raise ValueError("❌ AZURE_AI_FOUNDRY_CONNECTION_STRING is not set")
if not deployment_name:
    raise ValueError("❌ AZURE_OPENAI_CHAT_DEPLOYMENT_NAME is not set")

# Setup credentials
print("\n🔐 Setting up authentication...")
if os.environ.get("USE_AZURE_DEV_CLI") == "true":
    print("   ✅ Using Azure Developer CLI Credential")
    credential = AzureDeveloperCliCredential(tenant_id=tenant_id)
else:
    print("   ✅ Using Default Azure Credential")
    credential = DefaultAzureCredential()

# Initialize AIProjectClient for connection + agent discovery
print("\n🔧 Initializing AIProjectClient...")
project_client = AIProjectClient(endpoint=project_endpoint, credential=credential)
await project_client.__aenter__()
print("   ✅ AIProjectClient ready")

# Discover connections (same style as agents_v2 notebook)
print("\n" + "=" * 60)
print("🔗 AVAILABLE CONNECTIONS")
print("=" * 60)

model_gateway_connection_static = None
model_gateway_connection_dynamic = None
ai_gateway_connection_static = None
ai_gateway_connection_dynamic = None

connection_count = 0
async for connection in project_client.connections.list():
    connection_count += 1
    icon = (
        "🌐"
        if "ModelGateway" in str(connection.type)
        else "🔌" if "ApiManagement" in str(connection.type) else "📡"
    )
    default_badge = " ⭐ DEFAULT" if getattr(connection, "is_default", False) else ""
    print(f"\n{icon} {connection.name}{default_badge}")
    print(f"   Type: {connection.type}")
    print(f"   ID: {connection.id[:50]}...")

    type_text = str(connection.type)
    if "ModelGateway" in type_text and "static" in connection.name.lower():
        model_gateway_connection_static = connection.name
        print("   📌 → Assigned to: model_gateway_connection_static")
    if "ModelGateway" in type_text and "static" not in connection.name.lower():
        model_gateway_connection_dynamic = connection.name
        print("   📌 → Assigned to: model_gateway_connection_dynamic")
    if "ApiManagement" in type_text and "static" in connection.name.lower():
        ai_gateway_connection_static = connection.name
        print("   📌 → Assigned to: ai_gateway_connection_static")
    if "ApiManagement" in type_text and "static" not in connection.name.lower():
        ai_gateway_connection_dynamic = connection.name
        print("   📌 → Assigned to: ai_gateway_connection_dynamic")

print(f"\n📊 Total connections found: {connection_count}")

# List existing agents
print("\n" + "=" * 60)
print("🤖 EXISTING AGENTS")
print("=" * 60)

agents = []
async for existing_agent in project_client.agents.list():
    agents.append(existing_agent)

if agents:
    for existing_agent in agents:
        print(f"\n🤖 {existing_agent.name}")
        print(f"   ID: {existing_agent.id}")
        print(f"   Version: {existing_agent.versions.latest.version}")
else:
    print("\n   ℹ️ No agents found in this project")


def build_model_route(deployment: str, connection_name: str | None) -> str:
    return f"{connection_name}/{deployment}" if connection_name else deployment


static_connection_name = ai_gateway_connection_static or model_gateway_connection_static
dynamic_connection_name = ai_gateway_connection_dynamic or model_gateway_connection_dynamic

static_gateway_model = build_model_route(deployment_name, static_connection_name)
dynamic_gateway_model = build_model_route(deployment_name, dynamic_connection_name)

print("\n" + "=" * 60)
print("✅ SETUP COMPLETE - Connection Summary")
print("=" * 60)
print(
    f"   🌐 Model Gateway (Static):  {'✅ ' + model_gateway_connection_static if model_gateway_connection_static else '❌ Not found'}"
)
print(
    f"   🌐 Model Gateway (Dynamic): {'✅ ' + model_gateway_connection_dynamic if model_gateway_connection_dynamic else '❌ Not found'}"
)
print(
    f"   🔌 AI Gateway (Static):     {'✅ ' + ai_gateway_connection_static if ai_gateway_connection_static else '❌ Not found'}"
)
print(
    f"   🔌 AI Gateway (Dynamic):    {'✅ ' + ai_gateway_connection_dynamic if ai_gateway_connection_dynamic else '❌ Not found'}"
)
print(f"   🤖 Static model route:  {static_gateway_model}")
print(f"   🤖 Dynamic model route: {dynamic_gateway_model}")

# -- if not dynamic_connection_name:
# --     raise RuntimeError(
# --         "❌ No dynamic gateway connection found. Add a dynamic ApiManagement/ModelGateway connection before running dynamic tests."
# --     )

# Create clients for each route. Most tests use dynamic to mirror agents_v2 behavior.
foundry_client_static = FoundryChatClient(
    credential=credential,
    project_endpoint=project_endpoint,
    model=static_gateway_model,
)
foundry_client_dynamic = FoundryChatClient(
    credential=credential,
    project_endpoint=project_endpoint,
    model=dynamic_gateway_model,
)
foundry_client = foundry_client_dynamic
model = dynamic_gateway_model

print("\n🎉 Setup complete!")


async def cleanup():
    try:
        await project_client.close()
    except Exception:
        await project_client.__aexit__(None, None, None)
    try:
        await credential.close()
    except Exception:
        await credential.__aexit__(None, None, None)
    print("🧹 Clients closed")


def sync_cleanup():
    try:
        loop = asyncio.get_event_loop()
        if loop.is_running():
            loop.create_task(cleanup())
        else:
            loop.run_until_complete(cleanup())
    except Exception:
        pass


atexit.register(sync_cleanup)

## Create and Run a Basic Agent
Create an agent using `FoundryChatClient` and run it with a simple prompt.

In [ ]:
from agent_framework.foundry import FoundryAgent

print("🤖 BASIC AGENT TEST (STATIC GATEWAY)")
print("=" * 60)
print(f"   🔗 Connection: {static_connection_name}")
print(f"   🤖 Model route: {static_gateway_model}")
print()

# Create agent
#TODO: this does not create persistent agent - need to fix all agent creation to support persistence
agent = FoundryAgent(
    project_client=project_client,
    agent_name="MAF-BasicAgent",
    instructions="You are a helpful assistant that answers general questions.",
)
print(f"📝 Agent created: {agent.name}")

# Run agent
print("\n⏳ Running agent...")
response = await agent.run("What is the size of Poland in square miles?")

print("\n💬 Response:")
print("─" * 40)
print(response.text)
print("─" * 40)

## Streaming Response with Agent
Use `agent.run(..., stream=True)` to get real-time token-by-token output.

In [ ]:
print("🌊 STREAMING RESPONSE TEST")
print("=" * 60)
print("Testing real-time streaming of agent responses")
print()

# Create agent
agent = Agent(
    client=foundry_client_static,
    name="MAF-StreamingAgent",
    instructions="You are a helpful assistant.",
)
print(f"📝 Agent: {agent.name}")

# Stream the response
print("\n🌊 Streaming response:")
print("─" * 40)

stream = agent.run("Tell me hi in 10 random languages.", stream=True)
async for chunk in stream:
    if chunk.text:
        print(chunk.text, end="", flush=True)

final_response = await stream.get_final_response()

print("\n" + "─" * 40)
print("✅ Stream completed!")
print(f"\n📊 Final response messages: {len(final_response.messages)}")

## Agent with Function Tools
Create an agent with custom function tools. In the Microsoft Agent Framework,
tools are plain Python functions with type annotations — no special wrappers needed.

In [ ]:
from typing import Annotated
from pydantic import Field

print("🔧 FUNCTION TOOL CALLS")
print("=" * 60)
print("Testing agent with custom function tools")
print()


# Define user functions as plain Python functions with type annotations
def get_weather(
    location: Annotated[str, Field(description="The city name to get weather for.")],
) -> str:
    """Get the weather for a given location."""
    return f"The weather in {location} is 72°F and sunny."


def get_time(
    timezone: Annotated[str, Field(description="The timezone name (e.g., 'US/Eastern').")],
) -> str:
    """Get the current time in a given timezone."""
    from datetime import datetime
    return f"The current time in {timezone} is {datetime.now().strftime('%H:%M:%S')}."


print("🔌 Function tools configured:")
print("   • get_weather(location)")
print("   • get_time(timezone)")

# Create agent with function tools
print("\n📝 Creating agent with function tools...")
agent = Agent(
    client=foundry_client,
    name="MAF-FunctionToolAgent",
    instructions="You are a helpful agent. Use the provided tools to answer questions about weather and time.",
    tools=[get_weather, get_time],
)
print(f"   ✅ Agent: {agent.name}")

# Run with auto function calling
print("\n⏳ Running agent (auto function calls)...")
response = await agent.run("What's the weather in Seattle and what time is it in US/Eastern?")

print("\n💬 Response:")
print("─" * 40)
print(response.text)
print("─" * 40)

## Streaming with Function Tools
Stream responses while the SDK auto-executes function tool calls.

In [ ]:
print("🌊 STREAMING WITH FUNCTION TOOLS")
print("=" * 60)
print("Testing streaming + auto function call execution")
print()

# Create agent with tools
agent = Agent(
    client=foundry_client,
    name="MAF-StreamFuncAgent",
    instructions="You are a helpful agent. Use the provided tools to answer questions.",
    tools=[get_weather, get_time],
)
print(f"📝 Agent: {agent.name}")

# Stream with function tools
print("\n🌊 Streaming response (with auto tool calls):")
print("─" * 40)

stream = agent.run("What's the weather like in Paris and Tokyo?", stream=True)
async for chunk in stream:
    if chunk.text:
        print(chunk.text, end="", flush=True)

final_response = await stream.get_final_response()

print("\n" + "─" * 40)
print("✅ Stream completed!")
print(f"\n📊 Final response messages: {len(final_response.messages)}")

## Agent with Hosted MCP Tool
Use `FoundryChatClient.get_mcp_tool()` to configure a hosted MCP server
and pass it to the agent as a tool.

In [ ]:
print("🔧 HOSTED MCP TOOL CALLS")
print("=" * 60)
print("Testing agent with hosted MCP tool")
print()

# Configure hosted MCP tool using FoundryChatClient helper
mcp_tool = FoundryChatClient.get_mcp_tool(
    name="weather",
    url="https://aca-mcp-qczp34j2qg7pk.ashyocean-7ea49412.westus.azurecontainerapps.io/mcp/mcp",
    approval_mode="never_require",
)
print("🔌 MCP Tool configured:")
print(f"   Name: weather")
print(f"   Approval: never_require")

# Create agent with MCP tool
print("\n📝 Creating agent with MCP tool...")
agent = Agent(
    client=foundry_client,
    name="MAF-MCPAgent",
    instructions="You are a helpful agent that can use MCP tools to assist users. Use the available MCP tools to answer questions.",
    tools=[mcp_tool],
)
print(f"   ✅ Agent: {agent.name}")

# Run agent
print("\n⏳ Running agent...")
response = await agent.run("What's the forecast for Seattle?")

print("\n💬 Response:")
print("─" * 40)
print(response.text)
print("─" * 40)
print(f"\n📊 Response messages: {len(response.messages)}")

## Streaming with Hosted MCP Tool
Stream responses while the agent calls hosted MCP tools.

In [ ]:
print("🌊 STREAMING WITH HOSTED MCP TOOL")
print("=" * 60)
print("Testing streaming + hosted MCP tool execution")
print()

# Configure hosted MCP tool
mcp_tool = FoundryChatClient.get_mcp_tool(
    name="sample",
    url="https://sample-mcp-qczp34j2qg7pk.ashyocean-7ea49412.westus.azurecontainerapps.io/mcp",
    approval_mode="never_require",
)
print(f"🔌 MCP Tool: sample")

# Create agent with MCP tool
agent = Agent(
    client=foundry_client,
    name="MAF-MCPStreamAgent",
    instructions="You are a helpful agent that can use MCP tools to assist users.",
    tools=[mcp_tool],
)
print(f"📝 Agent: {agent.name}")

# Stream with MCP tool
print("\n🌊 Streaming response:")
print("─" * 40)

stream = agent.run("Say hello using a random name with MCP tool.", stream=True)
async for chunk in stream:
    if chunk.text:
        print(chunk.text, end="", flush=True)

final_response = await stream.get_final_response()

print("\n" + "─" * 40)
print("✅ Stream completed!")
print(f"\n📊 Final response messages: {len(final_response.messages)}")

## Agent Loop Test with Result Aggregation
Run multiple requests using an agent and capture:
- ✅ Success/Error status for each iteration
- 📊 Summary statistics and success rate
- 💾 Results exported to CSV

In [ ]:
import pandas as pd

print("🔧 AGENT LOOP TEST")
print("=" * 60)
print("Running multiple agent iterations to capture errors and success rates")
print()

# Create agent once
print("📝 Creating agent...")
agent = Agent(
    client=foundry_client,
    name="MAF-LoopTestAgent",
    instructions="You are a helpful assistant that answers general questions concisely.",
)
print(f"   ✅ Agent: {agent.name}")

# Configuration
num_iterations = 10
results = []

print(f"\n⚙️ Test Configuration:")
print(f"   • Iterations: {num_iterations}")
print(f"   • Model: {model}")
print(f"   • Agent: {agent.name}")
print()

for iteration in range(num_iterations):
    print(f"\n{'─' * 60}")
    print(f"🔄 Iteration {iteration + 1}/{num_iterations}")
    print(f"{'─' * 60}")

    result_status = None
    error_msg = None
    output_text = None

    try:
        # Run agent
        print("   ⏳ Running agent...")
        response = await agent.run(
            f"What is {iteration * 7 + 3} + {iteration * 11 + 5}? Reply with just the number."
        )

        output_text = response.text
        result_status = "SUCCESS"
        print(f"   ✅ SUCCESS")
        print(f"      Output: {output_text[:100] if output_text else 'No output'}")

    except Exception as e:
        result_status = "ERROR"
        error_msg = str(e)[:200]
        print(f"   ❌ ERROR: {error_msg}")

    results.append(
        {
            "Iteration": iteration + 1,
            "Result": result_status,
            "Output/Error": output_text if result_status == "SUCCESS" else error_msg,
        }
    )

# Create DataFrame and display results
df_results = pd.DataFrame(results)

print("\n" + "=" * 60)
print("📊 RESULTS TABLE")
print("=" * 60)
print(df_results.to_string(index=False))

# Save results to CSV
csv_file = "maf_agent_test_results.csv"
df_results.to_csv(csv_file, index=False)
print(f"\n💾 Results saved to: {csv_file}")

# Summary statistics
print("\n" + "=" * 60)
print("📈 SUMMARY")
print("=" * 60)
total = len(results)
successes = len([r for r in results if r["Result"] == "SUCCESS"])
errors = len([r for r in results if r["Result"] == "ERROR"])
success_rate = (successes / total * 100) if total > 0 else 0

print(f"   ✅ Successes: {successes}")
print(f"   ❌ Errors: {errors}")
print(f"   🎯 Success Rate: {success_rate:.1f}%")

if success_rate == 100:
    print("\n🎉 All tests passed!")
elif success_rate >= 50:
    print("\n⚠️ Some tests failed - check the results above")
else:
    print("\n❌ Most tests failed - investigate connection issues")

## MCP Agent Loop Test
Run multiple requests using an agent with hosted MCP tools and capture:
- ✅ Success/Error status for each iteration
- 📊 Summary statistics and success rate
- 💾 Results exported to CSV

In [ ]:
import pandas as pd

print("🔧 MCP AGENT LOOP TEST")
print("=" * 60)
print("Running MCP tools with agent - 10 iterations")
print()

# Configure MCP tool
mcp_tool = FoundryChatClient.get_mcp_tool(
    name="sample",
    url="https://sample-mcp-qczp34j2qg7pk.ashyocean-7ea49412.westus.azurecontainerapps.io/mcp",
    approval_mode="never_require",
)
print(f"🔌 MCP Tool: sample")

# Create agent once
print("\n📝 Creating agent...")
agent = Agent(
    client=foundry_client,
    name="MAF-MCPLoopAgent",
    instructions="You are a helpful agent that can use MCP tools to assist users. Use the available MCP tools to answer questions and perform tasks.",
    tools=[mcp_tool],
)
print(f"   ✅ Agent: {agent.name}")

# Configuration
num_iterations = 10
results = []

print(f"\n⚙️ Test Configuration:")
print(f"   • Iterations: {num_iterations}")
print(f"   • Model: {model}")
print(f"   • Agent: {agent.name}")
print()

for iteration in range(num_iterations):
    print(f"\n{'─' * 60}")
    print(f"🔄 Iteration {iteration + 1}/{num_iterations}")
    print(f"{'─' * 60}")

    result_status = None
    error_msg = None
    output_text = None

    try:
        # Run agent
        print("   ⏳ Running agent...")
        response = await agent.run("Say hello using a random name with MCP tool.")

        output_text = response.text
        result_status = "SUCCESS"
        print(f"   ✅ SUCCESS")
        print(f"      Output: {output_text[:500] if output_text else 'No output'}")

    except Exception as e:
        result_status = "ERROR"
        error_msg = str(e)[:200]
        print(f"   ❌ ERROR: {error_msg}")

    results.append(
        {
            "Iteration": iteration + 1,
            "Result": result_status,
            "Output/Error": output_text if result_status == "SUCCESS" else error_msg,
        }
    )

# Create DataFrame and display results
df_results = pd.DataFrame(results)

print("\n" + "=" * 60)
print("📊 RESULTS TABLE")
print("=" * 60)
print(df_results.to_string(index=False))

# Save results to CSV
csv_file = "maf_mcp_agent_test_results.csv"
df_results.to_csv(csv_file, index=False)
print(f"\n💾 Results saved to: {csv_file}")

# Summary statistics
print("\n" + "=" * 60)
print("📈 SUMMARY")
print("=" * 60)
total = len(results)
successes = len([r for r in results if r["Result"] == "SUCCESS"])
errors = len([r for r in results if r["Result"] == "ERROR"])
success_rate = (successes / total * 100) if total > 0 else 0

print(f"   ✅ Successes: {successes}")
print(f"   ❌ Errors: {errors}")
print(f"   🎯 Success Rate: {success_rate:.1f}%")

if success_rate == 100:
    print("\n🎉 All tests passed!")
elif success_rate >= 50:
    print("\n⚠️ Some tests failed - check the results above")
else:
    print("\n❌ Most tests failed - investigate connection issues")